# 01 - Préparer les données darija

Ce notebook prépare le corpus du chapitre 2. Nous utilisons uniquement les **données publiées par AtlasIA**, puis nous créerons notre propre tokenizer dans le notebook 02.

Parcours : choisir les CSV → inspecter la colonne `darija` → normaliser et dédupliquer → échantillonner → séparer entraînement/validation → sauvegarder les partitions.

**Le notebook est livré non exécuté.** Le téléchargement est désactivé par défaut ; aucune cellule de ce notebook n'entraîne de tokenizer ou de modèle.

## 1. Source et choix du premier corpus

Dataset : [atlasia/darija_english](https://huggingface.co/datasets/atlasia/darija_english). Pour commencer, nous choisissons `atlasia_stories.csv` et conservons sa colonne `darija`. Les traductions de la colonne `english` ne participent pas à l'apprentissage.

La page du dataset affiche la licence **CC BY-NC 4.0**. Les fichiers téléchargés restent locaux et sont exclus de Git. Nous fixons une révision du dataset pour identifier les données utilisées.

`MAX_DOCUMENTS` règle la taille de notre atelier. Il s'agit d'un paramètre pédagogique, pas d'une quantité suffisante garantie pour un LLM de bonne qualité.

In [ ]:
import csv
import hashlib
import json
import random
import unicodedata
from pathlib import Path
from urllib.request import urlopen

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / 'README.md').is_file() and (p / 'notebooks').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('Ouvre ce notebook depuis le dossier darija-llm-lab.')

DATASET = 'atlasia/darija_english'
REVISION = 'c6468fe948155c935c7a8ce59db7df895ad1296a'
SOURCE_FILES = ['atlasia_stories.csv']
MAX_DOCUMENTS = 1000
SEED = 42
VALIDATION_FRACTION = 0.10
DOWNLOAD_DATA = False
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
print('Projet :', ROOT)
print('CSV attendus dans :', RAW_DIR)

## 2. Mettre les CSV dans le projet

Deux possibilités : copier les fichiers déjà téléchargés dans `data/raw/`, ou passer `DOWNLOAD_DATA` à `True` puis exécuter la cellule suivante. Cette cellule réutilise un fichier existant sans le remplacer.

Pour identifier précisément un CSV copié manuellement, son hash SHA-256 sera enregistré. La révision demandée dans l'URL ne prouve pas à elle seule qu'un fichier local vient de cette révision.

In [ ]:
def source_url(filename):
    return (f'https://huggingface.co/datasets/{DATASET}/'
            f'resolve/{REVISION}/{filename}')

if DOWNLOAD_DATA:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for filename in SOURCE_FILES:
        destination = RAW_DIR / filename
        if destination.exists():
            print('Fichier réutilisé :', filename)
            continue
        with urlopen(source_url(filename), timeout=60) as response:
            payload = response.read()
        destination.write_bytes(payload)
        print('Téléchargé :', filename, '-', len(payload), 'octets')
else:
    print('Téléchargement désactivé. Utilise tes CSV locaux dans data/raw/.')

## 3. Inspecter les colonnes avant de transformer le texte

Un CSV peut contenir plusieurs informations : identifiant, traduction, source et texte darija. On vérifie le schéma au lieu de supposer que toutes les colonnes sont du texte à apprendre.

**Question :** pourquoi ne faut-il pas concaténer les traductions anglaises dans notre corpus darija sans décision explicite ?

In [ ]:
csv_paths = [RAW_DIR / name for name in SOURCE_FILES]
missing = [str(path) for path in csv_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'CSV manquants. Copie-les dans data/raw/ ou active DOWNLOAD_DATA : '
        + ', '.join(missing)
    )

for path in csv_paths:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if 'darija' not in (reader.fieldnames or []):
            raise ValueError(f'Colonne darija absente de {path.name}.')
        print(path.name, '→', reader.fieldnames)
        first_row = next(reader, None)
        if first_row:
            print('Premier extrait :', (first_row['darija'] or '')[:200])

## 4. Normaliser sans transformer la langue

Nous appliquons **NFC** et retirons les blancs aux bords des documents. NFC rapproche certaines représentations Unicode équivalentes. Elle ne transforme pas l'Arabizi en arabe et ne corrige pas les variantes orthographiques.

Exemples pédagogiques : `سلام، لاباس؟`, `salam, labas?`, `ghadi nji demain`. Les chiffres comme `3` et `7`, les accents et la ponctuation sont conservés.

Nous supprimons uniquement les doublons exacts après normalisation. Les paraphrases, doublons proches et documents apparentés demanderaient une analyse supplémentaire. La colonne `darija` peut aussi contenir du bruit ou d'autres langues : son nom n'est pas une garantie de qualité.

In [ ]:
def normalize_document(text):
    return unicodedata.normalize('NFC', text).strip()

def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def read_unique_documents(paths):
    documents, seen, sources = [], set(), []
    for path in paths:
        counts = {'rows': 0, 'empty_rows': 0, 'duplicate_rows': 0}
        with path.open(encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            if 'darija' not in (reader.fieldnames or []):
                raise ValueError(f'Colonne darija absente de {path.name}.')
            for row in reader:
                counts['rows'] += 1
                text = normalize_document(row.get('darija') or '')
                if not text:
                    counts['empty_rows'] += 1
                elif text in seen:
                    counts['duplicate_rows'] += 1
                else:
                    seen.add(text)
                    documents.append(text)
        sources.append({
            'filename': path.name, 'url': source_url(path.name),
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
            'bytes': path.stat().st_size, **counts,
        })
    return documents, sources

In [ ]:
if MAX_DOCUMENTS < 2 or not 0 < VALIDATION_FRACTION < 1:
    raise ValueError('Choisis au moins deux documents et une fraction entre 0 et 1.')

all_documents, source_manifest = read_unique_documents(csv_paths)
random.Random(SEED).shuffle(all_documents)
documents = all_documents[:MAX_DOCUMENTS]
if len(documents) < 2:
    raise ValueError('Il faut au moins deux documents distincts.')

n_valid = max(1, min(len(documents) - 1, round(len(documents) * VALIDATION_FRACTION)))
valid_texts = documents[:n_valid]
train_texts = documents[n_valid:]
assert set(train_texts).isdisjoint(valid_texts)

print('Documents distincts disponibles :', len(all_documents))
print('Documents sélectionnés :', len(documents))
print('Entraînement :', len(train_texts), '| Validation :', len(valid_texts))
for text in train_texts[:3]:
    print(repr(text[:160]))

## 5. Pourquoi séparer avant la tokenisation ?

Le tokenizer apprend son vocabulaire et ses fusions à partir de statistiques du corpus. La validation doit donc être réservée **avant** cet apprentissage. Les fenêtres glissantes seront construites séparément dans chaque partition au notebook 03.

Les partitions ci-dessus sont créées pour notre atelier ; elles ne correspondent pas à une partition d'évaluation officielle d'AtlasIA. Elles permettent de contrôler le protocole, sans garantir l'absence de doublons proches.

In [ ]:
def write_jsonl(path, texts):
    with path.open('w', encoding='utf-8', newline='\n') as stream:
        for text in texts:
            row = {'text': text, 'sha256': text_hash(text)}
            stream.write(json.dumps(row, ensure_ascii=False) + '\n')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(PROCESSED_DIR / 'train.jsonl', train_texts)
write_jsonl(PROCESSED_DIR / 'validation.jsonl', valid_texts)
corpus_manifest = {
    'dataset': DATASET, 'requested_revision': REVISION,
    'column': 'darija', 'dataset_license': 'CC-BY-NC-4.0',
    'sources': source_manifest, 'normalization': 'NFC + strip document edges',
    'deduplication': 'exact normalized documents across selected CSV files',
    'sampling': 'shuffle unique documents, then take the configured maximum',
    'seed': SEED, 'max_documents': MAX_DOCUMENTS,
    'available_unique_documents': len(all_documents),
    'train_documents': len(train_texts), 'validation_documents': len(valid_texts),
    'validation_used_for_tokenizer_training': False,
}
(PROCESSED_DIR / 'corpus_manifest.json').write_text(
    json.dumps(corpus_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Partitions sauvegardées dans :', PROCESSED_DIR)

## Exercices et points de contrôle

1. À quoi sert le hash du CSV ?
2. Pourquoi dédupliquer avant de séparer les documents ?
3. Est-ce que NFC traduit `salam` en `سلام` ?
4. Si tu changes les données ou la normalisation, quels notebooks faut-il relancer ?

<details><summary>Corrigés</summary>

1. Il permet d'identifier les octets exacts du fichier utilisé, même si le dataset évolue.
2. Un doublon exact ne doit pas se retrouver à la fois dans l'entraînement et la validation.
3. Non : NFC traite des équivalences Unicode, pas des équivalences de langue.
4. Relancer 01 pour recréer les partitions, puis 02 pour réapprendre le tokenizer, enfin 03 pour reconstruire les fenêtres.

</details>

**Suite :** [02 - Créer le tokenizer](02_creer_tokenizer_darija.ipynb).